# arXiv Paper Classification - MLP Baseline & BERT
___

As an introduction you can go to the section [Final conclusion](#final). There, I compared the models I've trained.

<a id="contents"></a>
# Contents

1. [Loading the dataset](#load-dataset)

2. [Exploratory Data Analysis](#eda)

    2.1. [Searching for missing values](#missing)
    
    2.2. [Analyzing the abstracts](#abstracts)
    
    2.3. [Analyzing the categories](#categories)
    
    2.4. [Analyzing paper uniqueness](#uniqueness)
    
    2.5. [Conclusion](#edaconclusion)
    
3. [Baseline model](#baseline)

    3.1. [Data preprocessing](#baseline-prep)
    
    3.2. [Model definition, training and evaluation](#baseline-model)
    
    3.3. [Baseline model inference](#baseline-infer)
    
    3.4. [Conclusion](#baseline-conclusion)
    
4. [Fine-tuning BERT](#bert)

    4.1. [Loading the dataset](#bert-loaddataset)
    
    4.2. [Tokenization](#bert-token)
    
    4.3. [Model definition](#bert-model)
    
    4.4. [Training and validation splits](#bert-splits)
    
    4.5. [Model training](#bert-training)
    
5. [Final conclusion](#final)

In [ ]:
import re
import ast
import time
import json
import pickle
import random
import inspect
import datetime as dt
import multiprocessing
from typing import Dict, Any, Generator, Tuple, Optional

import spacy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
import keras
import tensorflow_addons as tfa
from keras.layers import Dense, Dropout
from keras.models import Sequential
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    TFAutoModelForSequenceClassification,
    TFAutoModel,
    create_optimizer,
)
from huggingface_hub import notebook_login, KerasModelHubMixin


# Controlling the pseudo-randomness.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

In [ ]:
notebook_login()

<a id="load-dataset"></a>
# 1. Loading the dataset
[Back to Contents](#contents)
___

Each row in the input file (its path is denoted by `DATASET_PATH`) is a JSON with keys representing different field names (e.g. `abstract`, `classification`). Because of that, I'll go over its rows (using a `Generator` - not loading the full dataset into memory initially) and then create a Pandas DataFrame out of them.

The fields of this DataFrame will be:
- `date` - we might want to train a model over a smaller subset of the full dataset, based on date
- `abstract` - this field will be used for classification
- `categories` - this is the field with the labels. The objective is to do multi-labeled classification, since a single article may have multiple labels, while each one of them will have multiple classes

In [ ]:
DATASET_PATH = "/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json"

# The number of papers on which the models will be trained and that also will be analyzed.
# There are over 2 million papers in this dataset, I'll select a subset.
NUM_PAPERS = 5000

In [ ]:
def get_dataset_generator(path: str) -> Generator:
    with open(path, "r") as fp:
        for line in fp:
            row = json.loads(line)
            yield row

            
dataset_generator = get_dataset_generator(
    path=DATASET_PATH
)
print(type(dataset_generator))

In [ ]:
def create_dataframe(generator: Generator) -> pd.DataFrame:
    # I'll use this column to filter out paper duplicates.
    titles = []
    authors = []
    
    abstracts = []
    categories = []
    dates = []
    
    for row in generator:
        if len(abstracts) == NUM_PAPERS:
            break
        
        titles.append(row["title"])
        authors.append(row["authors"])
        
        dates.append(row["update_date"])
        abstracts.append(row["abstract"])
        categories.append(row["categories"])
        
    return pd.DataFrame.from_dict({
        "title": titles,
        "authors": authors,
        "date": dates,
        "abstract": abstracts, 
        "categories": categories
    })


dataset_df = create_dataframe(dataset_generator)
dataset_df["date"] = pd.to_datetime(dataset_df["date"])

<a id="eda"></a>
# 2. Exploratory Data Analysis
[Back to Contents](#contents)
___

The steps, through which I'll go over in this section, will be:
1. *Search for missing values* - the dataset might have missing abstracts or categories. I should decide what to do with the rows that have missing ones
2. *Analyzing the abstracts* - here, I'll measure the sizes (naively) of each abstract and create a length distribution of all of them. This'll help for when we truncate/pad the input text for the model(s)
3. *Analyzing the categories* - the values in this column will be our labels. I'll go over their frequency distribution, which might help in later decisions (i.e. which metrics to use). I'll also see how many unique classes there are - this will be used for categorization
4. *Analyzing the uniqueness of the papers* - if there are duplicates, they will be removed in the preprocessing step

In [ ]:
print(f"Dataset Samples: {len(dataset_df):,}")
dataset_df.head()

<a id="missing"></a>
## 2.1. Searching for missing values
[Back to Contents](#contents)

There are no missing values. We can continue without augmenting or removing rows.

In [ ]:
# Validating whether there are missing values or not.
print("Missing values by column:")
dataset_df.isnull().sum()

<a id="abstracts"></a>
## 2.2. Analyzing the abstracts
[Back to Contents](#contents)

As we can see, most of the abstracts have lengths of below $200$ words. That would mean that when we use the WordPiece Tokenizer for BERT, we might need to take an input of more than $200$ tokens (roughly), since the Tokenizer is a subword one.

For the Baseline model, I'll use TF-IDF vectorization.

In [ ]:
# I'll naively split the text by whitespace directly. It should be okay for an approximation.
# I might've alternatively tokenized using SpaCy, for example, or a HuggingFace WordPiece Tokenizer, for that matter.
dataset_df["abstract_len"] = dataset_df["abstract"].apply(lambda text: len(text.split()))
dataset_df["abstract_len"].hist()
plt.title("Abstract size distribution")
plt.show()

<a id="categories"></a>
## 2.3. Analyzing the categories
[Back to Contents](#contents)

There are $176$ unique paper categories (for all 2 million papers). 

The categories are mostly $1$ to $2$, per paper. 

The highest number of categories in a paper are $13$ (that's for all 2 million, for the first $5000$ papers, the highest number is $9$). It is a huge outlier, though, as we can see from the '*Number of categories per paper*' distribution and the description of the column.

The classes are far from balanced. This means that using *accuracy* would not be our best choice for a metric. 

I'll dive into the metric choosing later, before the baseline model training. Click [HERE](#baseline-model) to go to that section.

In [ ]:
# Splitting the string of space-separated categories into a list.
dataset_df["categories"] = dataset_df["categories"].apply(
    lambda text: tuple(text.split())
)
# Displaying the distribution of number of categories.
dataset_df["num_categories"] = dataset_df["categories"].apply(
    lambda cats: len(cats)
)
print("Number of categories per paper description:")
print(dataset_df["num_categories"].describe())

dataset_df["num_categories"].hist()
plt.title("Number of categories per paper")
plt.show()

In [ ]:
categories = dataset_df["categories"].tolist()

# Getting all unique categories by flattening the 'categories' column
# and creating a set out of the resultant list.
unique_categories = set([
    cat 
    for nested_cats in categories 
    for cat in nested_cats
])
print(f"Num. unique categories: {len(unique_categories)}")

In [ ]:
print(f"Earliest date: {dataset_df['date'].min()}")
print(f"Latest date: {dataset_df['date'].max()}")

all_categories = dataset_df["categories"].tolist()
all_categories = [
    cat 
    for nested_cats in all_categories 
    for cat in nested_cats
]

print(
    "Most frequently occurring category:", 
    max(set(all_categories), key=all_categories.count)
)
print(
    "Least frequently occurring category:", 
    min(set(all_categories), key=all_categories.count)
)

plt.hist(all_categories)
plt.tick_params(
    axis='x',
    which='both',
    bottom=False,
    top=False,
    labelbottom=False
)
plt.tick_params(
    axis='y',
    which='both',
    bottom=False,
    top=False,
    labelbottom=False
)
plt.title("Class frequency distribution")
plt.xlabel("Class")
plt.ylabel("Frequency")
plt.show()

<a id="uniqueness"></a>
## 2.4. Analyzing paper uniqueness
[Back to Contents](#contents)

I'll use a combination of `title` and `authors` as a uniqueness indicator.

In [ ]:
print(f"Num. papers: {len(dataset_df)}")
dataset_df = dataset_df[~dataset_df[["title", "authors"]].duplicated()]
print(f"Num. unique papers: {len(dataset_df)}")

In [ ]:
# Removing the unneeded columns for the training.
dataset_df.drop("title", axis=1, inplace=True)
dataset_df.drop("authors", axis=1, inplace=True)
dataset_df.drop("date", axis=1, inplace=True)
dataset_df.drop("abstract_len", axis=1, inplace=True)
dataset_df.drop("num_categories", axis=1, inplace=True)

<a id="edaconclusion"></a>
## 2.5. Conclusion
[Back to Contents](#contents)

Here, I'll summarize what are the next steps that will be taken, based on the EDA above (and not only).

1. For the Baseline model, I'll use a standard NLP preprocessing pipeline - special characters removal, word lowercasing, stopwords removal, stemming, and finally - vectorization using TF-IDF. BERT will take in the raw text data
2. Each article should be truncated or padded to size of approx. $200$ for BERT. That would suffice for topic understanding
3. I'll choose the number of labels to be equal to the number of unique categories. Since our objective is to make multi-label classification, the format of the output vector will be $[x_0, x_1, \dots, x_{176}]$, where $x_i$ is either $0$ or $1$.

<a id="baseline"></a>
# 3. Baseline model
[Back to Contents](#contents)
___

For a baseline model, I'll choose a simple *Multi-Layer Perceptron* (MLP).

For each label, the model will do a binary classification. In our case, that would be $176$ separate binary classifications. The output vector will be of size $\mathbb{R}^{176}$ (if we take all 2 million papers), where each element is either $0$ or $1$.

I won't delve that much into the hyperparameter tuning here, this model will be used for comparison.

Firstly, let's start with the preprocessing of the data.

<a id="baseline-prep"></a>
## 3.1. Data preprocessing
[Back to Contents](#contents)

In [ ]:
print(f"Dataset size: {len(dataset_df):,}")
print("Before preprocessing:")
dataset_df.head()

In [ ]:
dataset_df.to_csv("/kaggle/working/dataset.csv", sep="\t", index=False)

In [ ]:
%%time

class BaselinePreprocessor:
    """Here, I'll prepare the data for the multi-label classification of the
    baseline model. This class will preprocess the data in a parallel matter using 
    multiple processes. The original DataFrame will be split in chunks. A pipeline
    of transformations will be ran on each chunk.
    The pipeline of transformations will include:
    - Text lowercasing
    - Special symbols removal
    - Stopwords removal
    - Stemming (It is chosen for simplicity's sake, using lemmatization will be more resource intensive)
    
    Args:
        dataset (pandas.DataFrame): The dataset in a DataFrame format.
    """
    
    def __init__(self, dataset: pd.DataFrame):
        self._df_chunks = np.array_split(dataset, multiprocessing.cpu_count())
        self._spacy_nlp = spacy.load('en_core_web_sm')
        
        # Getting all defined methods.
        obj_methods = list(inspect.getmembers(self, predicate=inspect.ismethod))
        # Creating a pipeline of actions with only the '_step' methods.
        self._pipeline = [
            reference
            for method_name, reference in obj_methods 
            if "_step" in method_name
        ]
        
    def preprocess(self) -> pd.DataFrame:
        subdfs = []
                                         
        with multiprocessing.Pool() as pool:
            results = pool.map(self._run_pipeline, self._df_chunks)
        
        for result in results:
            subdfs.append(result)
                                         
        return pd.concat(subdfs)
                                         
    def _run_pipeline(self, df: pd.DataFrame) -> pd.DataFrame:
        for step in self._pipeline:
            df = step(df)
                                         
        return df
    
    def _step_lowercase(self, df: pd.DataFrame) -> pd.DataFrame:
        df["abstract"] = df["abstract"].apply(lambda text: text.lower())
        return df
        
    def _step_remove_special_symbols(self, df: pd.DataFrame) -> pd.DataFrame:
        df["abstract"] = df["abstract"].apply(self._remove_special_symbols)
        return df
        
    def _step_remove_stopwords(self, df) -> pd.DataFrame:
        df["abstract"] = df["abstract"].apply(self._remove_stopwords)
        return df
        
    def _step_stemming(self, df) -> pd.DataFrame:
        df["abstract"] = df["abstract"].apply(self._stemming)
        return df
        
    def _remove_stopwords(self, text: str) -> str:
        doc = self._spacy_nlp(text)
        
        filtered_words = [
            token.text 
            for token in doc 
            if not token.is_stop
        ]
        
        return " ".join(filtered_words)
    
    def _stemming(self, text: str) -> str:
        stemmer = PorterStemmer()
        stemmed_words = [
            stemmer.stem(token.text)
            for token in self._spacy_nlp(text)
        ]
        
        return " ".join(stemmed_words)
    
    def _remove_special_symbols(self, text: str) -> str:
        # Replacing symbols that are not letters, digits or whitespace with a space.
        text = re.sub(r"[^\w\d\s]+", " ", text)
        # Converting multiple whitespaces into single ones.
        text = re.sub(r"\s+", " ", text)
        
        return text.strip()


bl_preprocessor = BaselinePreprocessor(dataset_df)
prep_dataset_df = bl_preprocessor.preprocess()

print("After preprocessing:")
prep_dataset_df.head()

## Vectorization and binarization

In [ ]:
vectorizer = TfidfVectorizer()
mlb = MultiLabelBinarizer()


def vectorize_dataset(
    dataset: pd.DataFrame, 
    tfidf: TfidfVectorizer, binarizer: MultiLabelBinarizer,
    train: bool = False
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    if train:
        tfidf.fit(dataset["abstract"])
        binarizer.fit(dataset["categories"])
    
    X = tfidf.transform(dataset["abstract"])
    
    if train:
        y = binarizer.transform(dataset["categories"])
        return X, y

    return X


X, y = vectorize_dataset(
    dataset=prep_dataset_df,
    tfidf=vectorizer,
    binarizer=mlb,
    train=True
)

print(f"X.shape", X.shape)
print(f"y.shape", y.shape)

In [ ]:
# Saving the binarizer, so that we can (if this model is better) decode the outputs in the API.
with open("/kaggle/working/binarizer.pkl", "wb") as f:
    pickle.dump(mlb, f)
    
# Saving the vectorizer.
with open("/kaggle/working/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

## Training and validation split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, 
    test_size=0.2, random_state=SEED
)

# 

<a id="baseline-model"></a>
## 3.2. Model definition, training and evaluation
[Back to Contents](#contents)

The baseline model will be a simple MLP. The prediction task will be separated into multiple binary classifications. Since there are two types of classes for each label, I'll choose the `sigmoid` activation function as a final one (if there were more classes, I would've chosen the `softmax`).

> Please note, this model is only used for comparison. I won't be doing that much hyperparameter tuning.

In [ ]:
def create_mlp():
    mlp = Sequential()
    mlp.add(Dense(256, activation='relu'))
    mlp.add(Dense(len(mlb.classes_), activation='sigmoid'))
    
    return mlp


mlp = create_mlp()

For the training and evaluation I'll use:
- *Binary Cross-entropy* - because of the subtasks, which are binary classifications
- *Adam optimizer* - generally, a solid choice for an optimizer
- *Precision and F1-Score* - I'll explain my choice with an example. Let's say that the problem is a single-labeled binary classification. Our model predicts for an article to be either `maths` or `non-maths`-related. For *Recall* we'll minimze **False Negatives**, which means that **we'll try to predict most of the *original* *math* papers correctly**. For *Precision*, we'll minimize **False Positives**, which means that **we'll try to make our predictions as correct as possible**. For the user, it would be better to have his recommended `maths` papers to be as Maths-related as possible, rather than having a lot of Maths-related recommendations. I've also included F1-Score, since it would be great to also add Recall in the mix.

In [ ]:
# Here, I'll the define hyperparameters, loss function and optimizer of the Baseline MLP.
L_RATE = 1e-3
LOSS_FUNC = keras.losses.BinaryCrossentropy(from_logits=False)
OPTIMIZER = keras.optimizers.Adam(learning_rate=L_RATE)
EPOCHS = 15
BATCH_SIZE = 32

METRICS = [
    keras.metrics.Precision(name="Precision"),
    tfa.metrics.F1Score(
        name="F1-Score",
        num_classes=len(mlb.classes_),
        # Since the dataset is imbalanced, I'll use micro averaging.
        average="micro",
        threshold=0.5
    ),
]

In [ ]:
def make_prediction(
    model: Sequential, X: np.ndarray, threshold: float = 0.5
) -> np.ndarray:
    """Making a prediction with the Baseline MLP model.
    
    Args:
        model (Sequential): The model object.
        X (numpy.ndarray): The feature matrix.
        threshold (float): Rounding threshold.
    """
    y_pred = model.predict(X)
    return (y_pred > threshold).astype(int)


mlp.compile(
    loss=LOSS_FUNC,
    optimizer=OPTIMIZER,
    # I'll calculate Precision and F1-Score below, on the validation set.
    # This metric is just for good measure.
    metrics=METRICS,
)
X_train.sort_indices()
X_valid.sort_indices()

mlp.fit(
    X_train, 
    y_train,
    validation_data=(X_valid, y_valid),
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE
)


mlp.save_weights("baseline.h5")
print("Model saved!")

<a id="baseline-infer"></a>
## 3.3. Baseline model inference
[Back to Contents](#contents)

In [ ]:
X_infer = vectorize_dataset(
    dataset=prep_dataset_df.iloc[5:7],
    tfidf=vectorizer,
    binarizer=mlb,
    train=False
)
X_infer.sort_indices()

y_infer = make_prediction(model=mlp, X=X_infer)
print(vectorizer.inverse_transform(X_infer))
print(mlb.inverse_transform(y_infer))

<a id="baseline-conclusion"></a>
## 3.4. Conclusion
[Back to Contents](#contents)

With the simple MLP model, we've achieved precision of $89$% and F1-Score of $47$%.
Above you can see an example of how it predicts the categories of two papers.

Let's continue with the training of the Encoder-only Transformer - *BERT*.

<a id="bert"></a>
# 4. Fine-tuning BERT
[Back to Contents](#contents)

<a id="bert-loaddataset"></a>
## 4.1. Loading the dataset
[Back to Contents](#contents)

In [ ]:
dataset = load_dataset('csv', data_files="/kaggle/working/dataset.csv", sep="\t")
dataset

<a id="bert-token"></a>
## 4.2. Tokenization
[Back to Contents](#contents)

Using a *cased* version of BERT will mean a bigger vocabulary, which in our case will slow down the training and use up more memory. This is unnecessary - the categories of the paper can be found without the use of uppercase characters.

Therefore, I'll choose `bert-base-uncased`.

In [ ]:
MAX_SEQ_LEN = 200

MODEL_ID = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [ ]:
tokenizer.convert_ids_to_tokens(tokenizer("Hello there! General Kenobi!")["input_ids"])

Tokenizing examples in batches.

In [ ]:
def tokenize_example(examples):
    tokenized_output = tokenizer(
        examples["abstract"], 
        padding="max_length",
        return_tensors="np",
        max_length=MAX_SEQ_LEN,
        truncation=True
    )
    
    labels = [
        ast.literal_eval(cats) 
        for cats in examples["categories"]
    ]
    
    return {
        "input_ids": tokenized_output["input_ids"].tolist(),
        "attention_mask": tokenized_output["attention_mask"].tolist(),
        "labels": mlb.transform(labels),
    }


print("Original Abstract:")
print(dataset["train"][0]["abstract"])
tokenized_dataset = dataset.map(
    tokenize_example, 
    batched=True,
    remove_columns=["abstract", "categories"]
)
print("Tokenized Abstract:")
print(tokenized_dataset["train"][0]["input_ids"])

In [ ]:
# This is how I'll transform the predictions.
print(tokenized_dataset["train"]["labels"][:1])
mlb.inverse_transform(np.array(tokenized_dataset["train"]["labels"][:1]))

<a id="bert-model"></a>
## 4.3. Model definition
[Back to Contents](#contents)

I'll define and use two approaches for the usage of BERT:
- BERT with custom additional layers (ArxivBert) - I take the vectors corresponding to the first output token and pass them to the additional layers. The final additional layer has an `sigmoid` activation function.
- *TFAutoModelForSequenceClassification* for multi-labeled classification - BERT with a head that's compatible for multilabeled classification.

In [ ]:
class ArxivBert(keras.Model, KerasModelHubMixin):
    """Appending additional layers to the BERT model, so that it can be used
    for multi-label classification.
    """
    
    def __init__(self, base_model_id: str, num_labels: int):
        super().__init__()
        self._base = TFAutoModel.from_pretrained(base_model_id, from_pt=True)
        self._base.trainable = True
        
        self._additional_layers = keras.Sequential([
            Dropout(0.1),
            Dense(512, activation="relu"),
            Dense(256, activation="relu"),
            Dense(num_labels, activation="sigmoid"),
        ])
        
    def call(self, inputs):
        out = self._base(inputs)
        out = out["last_hidden_state"][:, 0, :]
        
        return self._additional_layers(out)

In [ ]:
IDS2LABELS = {idx: label for idx, label in enumerate(mlb.classes_)}
LABELS2IDS = {label: idx for idx, label in enumerate(mlb.classes_)}


# I'll try to use both of these models to see which one does better.
arxiv_bert_hf = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(mlb.classes_),
    id2label=IDS2LABELS,
    label2id=LABELS2IDS,
    problem_type="multi_label_classification"
)
arxiv_bert = ArxivBert(MODEL_ID, len(mlb.classes_))

<a id="bert-splits"></a>
## 4.4. Training and validation splits
[Back to Contents](#contents)


In [ ]:
tokenized_dataset = tokenized_dataset["train"].train_test_split(test_size=0.2)
train_dataset = tokenized_dataset["train"]
eval_dataset = tokenized_dataset["test"]
train_dataset

Here, I'll convert the HuggingFace datasets into TensorFlow ones.

In [ ]:
tf_train_dataset = arxiv_bert_hf.prepare_tf_dataset(
    train_dataset,
    batch_size=32,
    # It was already shuffled in the cell above.
    shuffle=False
)
tf_eval_dataset = arxiv_bert_hf.prepare_tf_dataset(
    eval_dataset,
    batch_size=32,
    # It was already shuffled in the cell above.
    shuffle=False
)

<a id="bert-training"></a>
## 4.5. Model training
[Back to Contents](#contents)

In the next two cells I tried to train the two BERT models (`arxiv_bert` and `arxiv_bert_hf`).

I had problems with `arxiv_bert_hf`. It had really bad results in comparison to my custom implementation of the additional layers. It had approximately $30$% precision and even lower F1-Score. I suppose, I was doing something wrong, but I couldn't figure out what.

As you can see below, the other one - `arxiv_bert_hf` has way better results. $90$% Precision and $72$% F1-Score while training and $64$% Precision and $50$% F1-Score on the validation set. Obviously, the model has **overfitted**. That could be fixed by *stratifying* the *train-validation split* - I haven't managed to do that yet.

In [ ]:
class EpochSaver(keras.callbacks.Callback):
    """Saving the model each epoch."""
    
    def on_epoch_end(self, epoch, logs={}):
        self.model.save("arxiv_bert.hd5")

In [ ]:
# One of the learning rates recommended in the original BERT paper.
L_RATE = 2e-5
EPOCHS = 15

OPTIMIZER = keras.optimizers.Adam(learning_rate=L_RATE)
LOSS_FUNC = keras.losses.BinaryCrossentropy(from_logits=False)


arxiv_bert.compile(
    loss=LOSS_FUNC,
    optimizer=OPTIMIZER,
    metrics=METRICS,
    jit_compile=True
)

arxiv_bert.fit(
    tf_train_dataset,
    validation_data=tf_eval_dataset,
    epochs=EPOCHS,
    # Used this line to save the model on each epoch.
    # callbacks=[EpochSaver(),]
)

In [ ]:
# This didn't work properly, after loading the model in the API,
# a lot of errors occurred.
# Pushing the tokenizer and the model to HuggingFace's Model Hub.
arxiv_bert.push_to_hub("arxiv-bert")
tokenizer.push_to_hub("arxiv-bert")

<a id="final"></a>
# 5. Final conclusion
[Back to Contents](#contents)

In this section I'll mention what was concluded after the training of all $3$ models.

[Here](#baseline-model), I've mentioned why I chose *Precision* and *F1-Score* for evaluation.

Model | Train. Precision | Train. F1-Score | Valid. Precision | Valid. F1-Score
---|---|---|---|---
Baseline (MLP) | 99% | 93% | 89% | 47%
ArxivBERT | 90% | 72% | 64% | 51%


#### Baseline (MLP)
The baseline MLP model did impressively well, for what it was. Its downsides were that the data had to be thoroughly preprocessed (see [BaselinePreprocessor](#baseline-prep)) and the vocabulary was restricted - because of TF-IDF vectorization.

#### ArxivBERT
ArxivBERT is the default uncased BERT (`bert-base-uncased`) with additional linear layers. 
I chose not to use automatic hyperparameter tuning. At first the model was underperforming even the results above - it was overfitting much more. Then, I added a Dropout layer to the model, and decreased the sizes of the additional Dense layers. This helped a lot with the increasing of the F1-Score (initially it was way below $51$%).
For greater regularization I could've also include *weight decay* in the Adam optimizer.

One more important thing is that *I did not stratify the training and validation splits*. This resulted in much greater overfitting. In hindsight, I would've stratified the split.

I also trained multi-label classification BERT **TFAutoModelForSequenceClassification**, but it performed poorly. I didn't manage to fix the problems with it.